In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("/Users/nidheeshpradeepkumar/Downloads/GitHub/Vendor Analysis/Vendor Analysis/data")
csv_files = sorted(data_dir.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(
        "No CSV files found in {data_dir.resolve()}. "
        "Put the extracted Kaggle CSV files in data/raw."
    )

Previous step: [01 — Data Inspection](01_data_inspection.ipynb)

In [3]:
import re
from pathlib import Path
import pandas as pd
for file_path in csv_files:
    df = pd.read_csv(file_path, low_memory=False)

# Keep the original df and raw CSV unchanged
clean_df = df.copy()
clean_df["source_row_index"] = clean_df.index

# Standardize headers, e.g. "PO / SO #" becomes "po_so"
clean_df.columns = [
    re.sub(r"[^a-z0-9]+", "_", str(column).lower()).strip("_")
    for column in clean_df.columns
]

numeric_columns = [
    "line_item_quantity",
    "line_item_value",
    "pack_price",
    "unit_price",
    "weight_kilograms",
    "freight_cost_usd",
    "line_item_insurance_usd",
]

exceptions = []

for column in numeric_columns:
    if column not in clean_df.columns:
        print(f"Column not found; check its actual header: {column}")
        continue

    original_values = clean_df[column].copy()
    converted_values = pd.to_numeric(original_values, errors="coerce")
    flagged = original_values.notna() & converted_values.isna()

    for row_index, raw_value in original_values[flagged].items():
        exceptions.append({
            "source_row_index": row_index,
            "po_so": clean_df.loc[row_index, "po_so"]
                if "po_so" in clean_df.columns else None,
            "field": column,
            "raw_value": raw_value,
            "issue": "Non-numeric value in numeric field",
        })

    # Convert only in the processed copy; original df stays unchanged
    clean_df[column] = converted_values

exceptions_df = pd.DataFrame(exceptions)

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(output_dir / "procurement_cleaned.csv", index=False)
exceptions_df.to_csv(output_dir / "data_quality_exceptions.csv", index=False)

print(f"Rows in original: {len(df):,}")
print(f"Rows in processed copy: {len(clean_df):,}")
print(f"Values flagged for review: {len(exceptions_df):,}")



Rows in original: 10,324
Rows in processed copy: 10,324
Values flagged for review: 8,078


In [4]:
numeric_cols = [
    "Line Item Quantity",
    "Line Item Value",
    "Pack Price",
    "Unit Price",
    "Weight (Kilograms)",
    "Freight Cost (USD)",
    "Line Item Insurance (USD)",
]

for column in numeric_cols:
    converted = pd.to_numeric(df[column], errors="coerce")
    flagged = df[df[column].notna() & converted.isna()]

    print(f"{column}: {len(flagged)} non-numeric, nonblank values")
    display(flagged[["PO / SO #", "ASN/DN #", column]].head())
    

Line Item Quantity: 0 non-numeric, nonblank values


,PO / SO #,ASN/DN #,Line Item Quantity


Line Item Value: 0 non-numeric, nonblank values


,PO / SO #,ASN/DN #,Line Item Value


Pack Price: 0 non-numeric, nonblank values


,PO / SO #,ASN/DN #,Pack Price


... (4 outputs left out)

Freight Cost (USD): 4126 non-numeric, nonblank values


,PO / SO #,ASN/DN #,Freight Cost (USD)
6,SCMS-139,ASN-130,Freight Included in Commodity Cost
8,SCMS-156,ASN-93,See ASN-93 (ID#:1281)
9,SCMS-165,ASN-199,Freight Included in Commodity Cost
10,SCMS-221,ASN-223,Freight Included in Commodity Cost
12,SCMS-230,ASN-144,Invoiced Separately


Line Item Insurance (USD): 0 non-numeric, nonblank values


,PO / SO #,ASN/DN #,Line Item Insurance (USD)


In [5]:
df_raw = df.copy(deep=True)
df_clean = df_raw.copy(deep=True)
df_clean.insert(0, "source_row_index", df_clean.index)

In [6]:
numeric_cols = [
    "Line Item Quantity",
    "Line Item Value",
    "Pack Price",
    "Unit Price",
    "Weight (Kilograms)",
    "Freight Cost (USD)",
    "Line Item Insurance (USD)",
]

exceptions = []

for column in numeric_cols:
    if column not in df_clean.columns:
        print(f"Column not found; check its exact name: {column}")
        continue

    raw_values = df_clean[column].astype("string").str.strip()
    parsed_values = pd.to_numeric(
        raw_values.str.replace(",", "", regex=False),
        errors="coerce"
    )

    invalid_mask = raw_values.notna() & raw_values.ne("") & parsed_values.isna()

    if invalid_mask.any():
        flagged = df_clean.loc[
            invalid_mask, ["source_row_index", "PO / SO #", "ASN/DN #", column]
        ].copy()
        flagged["issue"] = "Non-numeric value in numeric field"
        exceptions.append(flagged)

    df_clean[f"{column}_numeric"] = parsed_values

exceptions_df = (
    pd.concat(exceptions, ignore_index=True)
    if exceptions
    else pd.DataFrame()
)

print(f"Flagged non-numeric cells: {len(exceptions_df)}")


Flagged non-numeric cells: 8078


In [7]:
import numpy as np
import pandas as pd

weight_raw = df_clean["Weight (Kilograms)"].astype("string").str.strip()
weight_numeric = pd.to_numeric(weight_raw, errors="coerce")

df_clean["weight_kg_numeric"] = weight_numeric
df_clean["weight_status"] = np.select(
    [
        weight_numeric.notna(),
        weight_raw.str.match(r"(?i)^see\b", na=False),
        weight_raw.eq("Weight Captured Separately").fillna(False),
    ],
    [
        "Numeric value",
        "See ASN/DN reference",
        "Captured separately",
    ],
    default="Review other value",
)

freight_raw = df_clean["Freight Cost (USD)"].astype("string").str.strip()
freight_numeric = pd.to_numeric(freight_raw, errors="coerce")

df_clean["freight_usd_numeric"] = freight_numeric
df_clean["freight_status"] = np.select(
    [
        freight_numeric.notna(),
        freight_raw.str.match(r"(?i)^see\b", na=False),
        freight_raw.eq("Freight Included in Commodity Cost").fillna(False),
        freight_raw.eq("Invoiced Separately").fillna(False),
    ],
    [
        "Numeric value",
        "See ASN/DN reference",
        "Included in commodity cost",
        "Invoiced separately",
    ],
    default="Review other value",
)

In [8]:
display(df_clean["weight_status"].value_counts(dropna=False))
display(df_clean["freight_status"].value_counts(dropna=False))

weight_status
Numeric value           6372
See ASN/DN reference    2445
Captured separately     1507
Name: count, dtype: int64

freight_status
Numeric value                 6198
See ASN/DN reference          2445
Included in commodity cost    1442
Invoiced separately            239
Name: count, dtype: int64

In [9]:
date_cols = [
    "PQ First Sent to Client Date",
    "PO Sent to Vendor Date",
    "Scheduled Delivery Date",
    "Delivered to Client Date",
    "Delivery Recorded Date",
]

date_issues = []

for column in date_cols:
    if column not in df_clean.columns:
        print(f"Column not found; check its exact name: {column}")
        continue

    raw_dates = df_clean[column].astype("string").str.strip()
    parsed_dates = pd.to_datetime(raw_dates, errors="coerce")

    invalid = raw_dates.notna() & raw_dates.ne("") & parsed_dates.isna()

    # Save parsed dates separately; keep the original source column
    df_clean[f"{column}_parsed"] = parsed_dates

    print(
        f"{column}: {parsed_dates.notna().sum()} parsed; "
        f"{invalid.sum()} nonblank values need review"
    )

    if invalid.any():
        date_issues.append(
            df_clean.loc[invalid, ["PO / SO #", column]]
                    .assign(issue="Date could not be parsed")
        )

date_issues_df = (
    pd.concat(date_issues, ignore_index=True)
    if date_issues
    else pd.DataFrame()
)


PQ First Sent to Client Date: 7643 parsed; 2681 nonblank values need review
PO Sent to Vendor Date: 4592 parsed; 5732 nonblank values need review
Scheduled Delivery Date: 10324 parsed; 0 nonblank values need review
Delivered to Client Date: 10324 parsed; 0 nonblank values need review


/var/folders/20/sw4qbjxj62qdydg96yd6sbs40000gn/T/ipykernel_85720/3097234157.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(raw_dates, errors="coerce")
/var/folders/20/sw4qbjxj62qdydg96yd6sbs40000gn/T/ipykernel_85720/3097234157.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(raw_dates, errors="coerce")
/var/folders/20/sw4qbjxj62qdydg96yd6sbs40000gn/T/ipykernel_85720/3097234157.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(raw_dates, errors="coerce")
/var/folders/20/sw4

Delivery Recorded Date: 10324 parsed; 0 nonblank values need review


In [10]:
pq_col = "PQ First Sent to Client Date"
po_col = "PO Sent to Vendor Date"

pq_raw = df_raw[pq_col].astype("string").str.strip()
po_raw = df_raw[po_col].astype("string").str.strip()

df_clean["pq_date_status"] = np.select(
    [
        df_clean[f"{pq_col}_parsed"].notna(),
        pq_raw.eq("Pre-PQ Process").fillna(False),
        pq_raw.eq("Date Not Captured").fillna(False),
    ],
    ["Date captured", "Pre-PQ Process", "Date not captured"],
    default="Review other value",
)

df_clean["po_date_status"] = np.select(
    [
        df_clean[f"{po_col}_parsed"].notna(),
        po_raw.eq("N/A - From RDC").fillna(False),
        po_raw.eq("Date Not Captured").fillna(False),
    ],
    ["Date captured", "Not applicable - From RDC", "Date not captured"],
    default="Review other value",
)

display(df_clean["pq_date_status"].value_counts())
display(df_clean["po_date_status"].value_counts())

pq_date_status
Date captured        7643
Pre-PQ Process       2476
Date not captured     205
Name: count, dtype: int64

po_date_status
Not applicable - From RDC    5404
Date captured                4592
Date not captured             328
Name: count, dtype: int64

In [11]:
scheduled = df_clean["Scheduled Delivery Date_parsed"]
delivered = df_clean["Delivered to Client Date_parsed"]
recorded = df_clean["Delivery Recorded Date_parsed"]

df_clean["delivery_variance_days"] = (delivered - scheduled).dt.days

df_clean["delivery_status"] = np.select(
    [
        df_clean["delivery_variance_days"] < 0,
        df_clean["delivery_variance_days"] == 0,
        df_clean["delivery_variance_days"] > 0,
    ],
    ["Early", "On scheduled date", "Late"],
    default="Review date",
)

# Review whether any delivery was recorded before it actually occurred
recorded_before_delivery = df_clean[recorded < delivered]

print(df_clean["delivery_status"].value_counts(dropna=False))
print(f"Recorded before delivered date: {len(recorded_before_delivery)}")
display(
    recorded_before_delivery[
        ["PO / SO #", "ASN/DN #",
         "Delivered to Client Date_parsed",
         "Delivery Recorded Date_parsed"]
    ].head(20)
)

delivery_status
On scheduled date    6324
Early                2814
Late                 1186
Name: count, dtype: int64
Recorded before delivered date: 3


,PO / SO #,ASN/DN #,Delivered to Client Date_parsed,Delivery Recorded Date_parsed
3601,SCMS-199730,ASN-22935,2013-12-24,2013-12-23
5155,SCMS-210050,ASN-25327,2014-05-15,2014-05-14
6051,SCMS-204300,ASN-21709,2013-10-30,2013-10-29


In [12]:
scheduled = df_clean["Scheduled Delivery Date_parsed"]
delivered = df_clean["Delivered to Client Date_parsed"]
recorded = df_clean["Delivery Recorded Date_parsed"]

df_clean["delivery_variance_days"] = (delivered - scheduled).dt.days

df_clean["delivery_status"] = np.select(
    [
        df_clean["delivery_variance_days"] < 0,
        df_clean["delivery_variance_days"] == 0,
        df_clean["delivery_variance_days"] > 0,
    ],
    ["Early", "On scheduled date", "Late"],
    default="Review date",
)

# Review whether any delivery was recorded before it actually occurred
recorded_before_delivery = df_clean[recorded < delivered]

print(df_clean["delivery_status"].value_counts(dropna=False))
print(f"Recorded before delivered date: {len(recorded_before_delivery)}")

display(
    recorded_before_delivery[
        ["PO / SO #", "ASN/DN #",
         "Delivered to Client Date_parsed",
         "Delivery Recorded Date_parsed"]
    ].head(20)
)

delivery_status
On scheduled date    6324
Early                2814
Late                 1186
Name: count, dtype: int64
Recorded before delivered date: 3


,PO / SO #,ASN/DN #,Delivered to Client Date_parsed,Delivery Recorded Date_parsed
3601,SCMS-199730,ASN-22935,2013-12-24,2013-12-23
5155,SCMS-210050,ASN-25327,2014-05-15,2014-05-14
6051,SCMS-204300,ASN-21709,2013-10-30,2013-10-29


In [13]:
fields_to_check = [
    "Line Item Quantity",
    "Line Item Value",
    "Pack Price",
    "Unit Price",
    "Line Item Insurance (USD)",
]

for field in fields_to_check:
    numeric_field = f"{field}_numeric"

    if numeric_field not in df_clean.columns:
        print(f"Parsed column not found: {numeric_field}")
        continue

    values = df_clean[numeric_field]
    flagged = values.notna() & (values <= 0)

    print(f"{field}: {flagged.sum()} zero or negative values")

    

Line Item Quantity: 0 zero or negative values
Line Item Value: 17 zero or negative values
Pack Price: 18 zero or negative values
Unit Price: 103 zero or negative values
Line Item Insurance (USD): 54 zero or negative values


In [14]:
fields_to_check = [
    "Line Item Value",
    "Pack Price",
    "Unit Price",
    "Line Item Insurance (USD)",
]

for field in fields_to_check:
    numeric_field = f"{field}_numeric"
    values = df_clean[numeric_field]

    print(
        f"{field}: "
        f"{(values == 0).sum()} zero, "
        f"{(values < 0).sum()} negative"
    )

    flagged = values.notna() & (values <= 0)

    detail_cols = [
        "source_row_index",
        "ID",
        "Vendor",
        "PO / SO #",
        "ASN/DN #",
        "Item Description",
        "Line Item Quantity_numeric",
        "Line Item Value_numeric",
        "Pack Price_numeric",
        "Unit Price_numeric",
        "Line Item Insurance (USD)_numeric",
    ]
    detail_cols = [col for col in detail_cols if col in df_clean.columns]



Line Item Value: 17 zero, 0 negative
Pack Price: 18 zero, 0 negative
Unit Price: 103 zero, 0 negative
Line Item Insurance (USD): 54 zero, 0 negative


In [15]:
import re
import sqlite3
from pathlib import Path

sql_df = df_clean.copy()

# Convert headers like "PO / SO #" to SQL-friendly names like "po_so"
sql_df.columns = [
    re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", col.strip().lower())).strip("_")
    for col in sql_df.columns
]

if sql_df.columns.duplicated().any():
    raise ValueError("Two column names became identical during SQL name cleanup.")

db_path = Path("data/processed/vendor_analysis.db")
db_path.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(db_path)
sql_df.to_sql("stg_shipments", conn, if_exists="replace", index=False)

print(f"Loaded {len(sql_df):,} rows into {db_path}")

Loaded 10,324 rows into data/processed/vendor_analysis.db
